# 01 — Turning Transactions into Tokens

**By the end of this notebook** you'll understand why financial transaction data needs its own tokenisation scheme, and you'll have watched the real PRAGMA tokenisers process a small set of synthetic transactions field by field.

## What this notebook teaches

- Why you can't just treat a transaction row as a text string (§2.2)
- The four field-type strategies PRAGMA uses: numerical (percentile buckets), categorical (single token), text (BPE subwords), and temporal (log-seconds + calendar features)
- How to `fit()` each tokeniser on training data and `encode()` a new value
- What the vocabulary breakdown looks like across field types
- What a single transaction looks like after tokenisation — the `(key, value, time)` shape

## Jupyter primer — read this if you've never used Jupyter before

A Jupyter notebook is a document made of **cells**. There are two kinds:

- **Markdown cells** (like this one) contain formatted text. They explain what's happening.
- **Code cells** (grey background, with `[ ]:` on the left) contain Python. They run.

**To run a cell:** click on it and press **Shift+Enter**. The output appears directly below the cell.

**The `[*]` indicator:** while a cell is running, the bracket shows `[*]`. When it finishes it shows a number like `[1]`. If you see `[*]` for more than a few seconds the cell is still working — wait for it.

**Kernel state:** Python variables persist between cells. If you run cell 5 before cell 3 you might get a `NameError`. Always run cells in order, top to bottom. If something seems broken, go to **Kernel → Restart & Run All** to start fresh.

**Reading output:** output appears below the code cell that produced it. A `print()` call produces text. A matplotlib chart appears as an image. A pandas DataFrame appears as a formatted table.

**That's all you need.** Run this notebook top to bottom with Shift+Enter and you'll be fine.

## Prerequisites

No previous notebooks required — this is the first one.

**How to use:** run every cell in order with Shift+Enter. The setup cell below puts the repo on your Python path so the imports resolve correctly.

In [ ]:
# ── Setup: repo root on sys.path ─────────────────────────────────────────────
import sys, pathlib
repo_root = pathlib.Path().resolve().parent   # notebooks/ → repo root
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Standard library ──────────────────────────────────────────────────────────
import math
from datetime import datetime, timezone

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np                 # numerical arrays
import pandas as pd                # tabular display
import matplotlib.pyplot as plt    # visualisations
import matplotlib

# ── PRAGMA tokenisers (src/pragma_encoder/tokenizer/) ─────────────────────────
from pragma_encoder.tokenizer.numerical   import NumericalTokenizer   # percentile buckets
from pragma_encoder.tokenizer.categorical import CategoricalTokenizer  # one token per value
from pragma_encoder.tokenizer.textual     import TextualTokenizer      # BPE subwords
from pragma_encoder.tokenizer.temporal    import TemporalTokenizer     # log-seconds + calendar

matplotlib.rcParams['figure.dpi'] = 110
print("Setup complete. PRAGMA tokenisers imported.")

## Why transactions need their own tokenisation

Language models tokenise text by splitting it into subwords. That works because text is a sequence of characters with no inherent structure. A bank transaction is different — it's a row of typed columns:

| amount | merchant_category | merchant_name | timestamp |
|--------|------------------|--------------|-----------|
| 47.80 | GROCERY | Woolworths | 2024-03-15 08:22 |

If you naively serialise this as `"amount=47.80 merchant_category=GROCERY ..."` and run it through a text BPE tokeniser, you lose the field structure entirely. The model has to re-learn that `47` and `.80` belong to the same numerical field, that `GROCERY` is a categorical label, and that the timestamp encodes periodicity (morning, Friday, mid-month).

PRAGMA's solution (§2.2) is **key-value-time tokenisation**: each field gets its own tokeniser matched to its data type. The output is a structured `(key, value, time)` sequence — not a flat string. Four field types, four strategies.

## Build synthetic training data

We'll create 50 synthetic transactions to fit the tokenisers on. These aren't real — just plausible values to demonstrate fitting and encoding.

In [ ]:
rng = np.random.default_rng(42)
n = 50

amounts = np.concatenate([
    np.zeros(5),                          # some exact zeros (§2.2: extra bucket for zero)
    rng.exponential(scale=80, size=45),   # heavy-tailed, like real transaction amounts
])

categories = rng.choice(
    ["GROCERY", "TRANSPORT", "RESTAURANT", "ONLINE", "ATM", "UTILITY"], size=n)

merchants = rng.choice(
    ["Woolworths", "PayPal", "Uber", "Amazon", "Shell", "Netflix", "BP", "Coles"], size=n)

base_ts = datetime(2024, 3, 1, tzinfo=timezone.utc).timestamp()
timestamps = base_ts + rng.uniform(0, 30 * 24 * 3600, size=n)

df = pd.DataFrame({
    "amount":             amounts,
    "merchant_category":  categories,
    "merchant_name":      merchants,
    "timestamp":          timestamps,
})

print(f"Synthetic transactions: {len(df)} rows")
df.head(6)

## Strategy 1 — Numerical fields: percentile buckets

Transaction amounts are continuous and heavy-tailed. A $5 coffee and a $5,000 flight are both valid values. PRAGMA's solution is to **discretise into percentile buckets** (§2.2): fit N equally-spaced percentile boundaries on the training distribution, then map each value to the bucket it falls into. The model learns one embedding per bucket.

One special rule: **exactly zero gets its own dedicated bucket** (§2.2: "extra bucket for zero"). A zero fee is semantically distinct from a very small fee, so the model shouldn't confuse them.

Source: `src/pragma_encoder/tokenizer/numerical.py::NumericalTokenizer`

In [ ]:
num_tok = NumericalTokenizer(n_buckets=10)   # 10 buckets — small for readability
num_tok.fit(df["amount"].tolist())

print(f"Vocabulary: {num_tok.vocab_size} tokens")
print(f"  Buckets 0–{num_tok.n_buckets - 1}: percentile ranges")
print(f"  Bucket {num_tok.ZERO_ID}:   ZERO (exact 0.0, §2.2)")
print(f"  Bucket {num_tok.MISSING_ID}: MISSING (NaN/None)")
print()

for val in [0.0, 5.50, 47.80, 250.0, None]:
    tid = num_tok.encode(val)
    mid = num_tok.decode(tid)
    tag = "ZERO" if tid == num_tok.ZERO_ID else (
          "MISSING" if tid == num_tok.MISSING_ID else f"bucket {tid}")
    print(f"  amount={str(val):>8}  → token {tid:>2d} ({tag:>10})  midpoint≈{mid:.2f}")

## Strategy 2 — Categorical fields: single token

Merchant category codes, currencies, transaction types — these are categorical values. Each unique value gets one token. Unseen values at inference time map to `[UNK]`. That's it — simple, and effective because the embedding table learns a representation for each category from the training data.

Source: `src/pragma_encoder/tokenizer/categorical.py::CategoricalTokenizer`

In [ ]:
cat_tok = CategoricalTokenizer()
cat_tok.fit(df["merchant_category"].tolist())

print(f"Vocabulary: {cat_tok.vocab_size} tokens  ([UNK]=0, [MISSING]=1, then one per category)")
print()

for val in ["GROCERY", "TRANSPORT", "ONLINE", "CASINO", None]:   # CASINO is unseen
    tid      = cat_tok.encode(val)
    decoded  = cat_tok.decode(tid)
    print(f"  category={str(val):>12}  → token {tid:>2d}  decoded='{decoded}'")

## Strategy 3 — Text fields: BPE subwords

Merchant names are free text — `"WOOLWORTHS"`, `"Amazon Prime*XY7Z"`, `"SHELL BONDI BEACH"`. A fixed categorical vocabulary can't cover them all. PRAGMA uses **Byte-Pair Encoding (BPE)** (§2.2) — the same algorithm used in GPT and BERT. BPE learns common character sequences from the training corpus and builds a vocabulary of subword pieces. An unseen merchant name is split into known subwords.

Source: `src/pragma_encoder/tokenizer/textual.py::TextualTokenizer`

In [ ]:
# The tokenizers library is optional — guard with try/except
try:
    txt_tok = TextualTokenizer(vocab_size=200, max_length=4)
    txt_tok.fit(df["merchant_name"].tolist())

    print(f"BPE vocabulary: {txt_tok.vocab_size} subword tokens, max {txt_tok.max_length} per field")
    print()
    for name in ["Woolworths", "PayPal", "IKEA Sydney CBD"]:
        ids     = txt_tok.encode(name)
        decoded = txt_tok.decode(ids)
        print(f"  '{name}'  →  IDs {ids}  →  decoded: '{decoded}'")

    _txt_fitted = True
    _txt_vocab  = txt_tok.vocab_size
except ImportError:
    print("'tokenizers' library not available — showing concept only.")
    print("Each merchant name is split into learned subword pieces.")
    print("Install with: pip install tokenizers")
    _txt_fitted = False
    _txt_vocab  = 200   # the size we would have used

## Strategy 4 — Temporal fields: log-seconds + calendar features

Timestamps carry two distinct kinds of information:

1. **How long ago?** Gaps range from seconds to months. PRAGMA compresses this range with Equation 2 (§2.2): `t' = 8·ln(1 + t/8)`. The constant 8 keeps short gaps linear while squishing long gaps logarithmically.

2. **When in the calendar?** A 2am Sunday transaction has a different risk profile to a 10am weekday one. PRAGMA extracts exactly three calendar features: hour-of-day, day-of-week, day-of-month (§2.2). Month and quarter are not used.

The log-seconds coordinate feeds RoPE positional encoding. Calendar features feed a small MLP inside the Event Encoder (§2.3.3). Two separate pathways.

Source: `src/pragma_encoder/tokenizer/temporal.py::TemporalTokenizer`

In [ ]:
temp_tok = TemporalTokenizer()

print("Log-seconds coordinate  t' = 8·ln(1 + t/8)  (Equation 2, §2.2)")
for label, gap in [("0 s", 0), ("1 min", 60), ("1 hour", 3600),
                   ("1 day", 86400), ("1 week", 604800), ("30 days", 2592000)]:
    tp = temp_tok.compute_temporal_coordinate(gap)
    print(f"  gap={label:>8}  →  t' = {tp:7.3f}")

print()
print("Calendar features  [hour, day_of_week, day_of_month]  (§2.2: exactly 3)")
ts_example = datetime(2024, 3, 15, 8, 22, tzinfo=timezone.utc)   # Friday morning
cal = temp_tok.extract_calendar_features(ts_example)
print(f"  {ts_example.strftime('%Y-%m-%d %H:%M')}  →  "
      f"hour={cal[0]:.0f}, day_of_week={cal[1]:.0f} (0=Mon), day_of_month={cal[2]:.0f}")

## Visualisation 1 — Vocabulary breakdown by field type

In [ ]:
labels = ["Numerical\n(amount)", "Categorical\n(merch. cat.)",
          "Text / BPE\n(merchant name)", "Temporal\n(calendar)"]
sizes  = [num_tok.vocab_size, cat_tok.vocab_size, _txt_vocab, temp_tok.vocab_size]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, sizes, color=colors, width=0.5, edgecolor="white", linewidth=1.5)

for bar, sz in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(sz), ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Vocabulary size (tokens)", fontsize=11)
ax.set_title("PRAGMA vocabulary breakdown by field type (§2.2)", fontsize=12)
ax.set_ylim(0, max(sizes) * 1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**What you're looking at:** each bar shows how many tokens one field type contributes to the shared PRAGMA vocabulary. BPE fields need the most coverage because merchant names are open-ended. In a real PRAGMA deployment the value vocabulary runs to ~28,000 tokens total (§2.2).

## Visualisation 2 — The log-seconds compression in action

In [ ]:
t_raw   = np.logspace(0, 7, 500)           # 1 second → ~4 months, log-spaced
t_prime = np.array([temp_tok.compute_temporal_coordinate(t) for t in t_raw])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t_raw / 3600, t_prime, color="#4C72B0", linewidth=2)
ax1.set_xlabel("Elapsed time (hours)", fontsize=10)
ax1.set_ylabel("t'  =  8·ln(1 + t / 8)", fontsize=10)
ax1.set_title("Linear x-axis", fontsize=11)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax2.semilogx(t_raw, t_prime, color="#DD8452", linewidth=2)
ax2.set_xlabel("Elapsed time (seconds, log scale)", fontsize=10)
ax2.set_ylabel("t'  =  8·ln(1 + t / 8)", fontsize=10)
ax2.set_title("Log x-axis", fontsize=11)
for lbl, sec in [("1 min", 60), ("1 hr", 3600), ("1 day", 86400), ("1 wk", 604800)]:
    tp = temp_tok.compute_temporal_coordinate(sec)
    ax2.annotate(lbl, xy=(sec, tp), xytext=(sec * 2, tp + 0.8), fontsize=8, color="#444")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.suptitle("Equation 2 (§2.2): temporal coordinate compresses gap scale", fontsize=12)
plt.tight_layout()
plt.show()

**What you're looking at:** the left plot shows t' flattening as gaps grow — that's the log compression. Short gaps stay distinguishable; month-long gaps don't drown everything else. On the log x-axis (right), the curve looks nearly linear, which is why RoPE can use it as a smooth positional coordinate.

## A single transaction as (key, value, time) tokens

Let's take one row from our DataFrame and show exactly what PRAGMA sees after tokenisation.

In [ ]:
row = df.iloc[3]
ts  = datetime.fromtimestamp(row["timestamp"], tz=timezone.utc)

# Key names are symbolic here — in the real pipeline FinancialTokenizerPipeline
# assigns a numeric key ID per field name.
fields = [
    ("amount",            f"{row['amount']:.2f}",         num_tok.encode(row["amount"])),
    ("merchant_category", row["merchant_category"],      cat_tok.encode(row["merchant_category"])),
    ("merchant_name",     row["merchant_name"],
     txt_tok.encode(row["merchant_name"])[0] if _txt_fitted else "(BPE n/a)"),
    ("timestamp",         ts.strftime("%Y-%m-%d %H:%M"),  temp_tok.encode(ts)[0]),
]

# Time coordinate for this event (elapsed since event = 0)
t_coord = temp_tok.compute_temporal_coordinate(0.0)

token_df = pd.DataFrame([
    {"Field (key)": k, "Raw value": v, "Value token": tok, "Time t'": f"{t_coord:.3f}"}
    for k, v, tok in fields
])

print(f"Transaction: {dict(row)}")
print()
token_df

**What just happened:** each field in the transaction has been encoded into a small integer (`Value token`). The model looks up a learned embedding vector for each of those integers. The `Time t'` column is the log-seconds coordinate — here it's 0.0 because we're measuring from the event itself. In a customer's history of N events, each event gets a distinct t' based on how long before the most recent event it occurred.

## What just happened — section recap

You've just watched four different tokenisers process the same transaction:

- **NumericalTokenizer** mapped the raw amount to a percentile bucket — with a dedicated slot for exact zeros
- **CategoricalTokenizer** mapped the merchant category to a single integer
- **TextualTokenizer** split the merchant name into BPE subword IDs
- **TemporalTokenizer** computed the log-seconds coordinate and extracted three calendar features

The result is a sequence of small integers — the input representation that flows into PRAGMA's three encoders.

## Closing — what you now know

**Tokenisation:** PRAGMA assigns each transaction field to one of four strategies matched to its data type. The output is a structured `(key, value, time)` sequence rather than a flat text string.

**The four strategies at a glance:**
| Field type | Strategy | Vocab |
|---|---|---|
| Numerical | Percentile buckets + zero bucket | n_buckets + 2 |
| Categorical | One token per unique value | n_categories + 2 |
| Text | BPE subwords | 8,000–28,000 |
| Temporal | Log-seconds + 3 calendar features | 62 |

**Next:** notebook 02 shows what the model does with these tokens — the three-encoder architecture and the masked event modelling pretraining loop.

**Going deeper:** `docs/paper-to-code.md §2.2` maps every tokeniser to the paper section that specifies it. `src/pragma_encoder/tokenizer/financial_pipeline.py` shows how the four tokenisers are wired together for IBM TabFormer data.